## Text input

https://platform.openai.com/docs/models

In [8]:
import os
import sys
from dotenv import load_dotenv

# Add parent directory to path to import azure_openai_config
sys.path.append(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', '..'))

load_dotenv()

from azure_openai_config import setup_azure_openai, get_azure_openai_model
setup_azure_openai()

In [9]:
from langchain.agents import create_agent

agent = create_agent(
    model=get_azure_openai_model(),
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [10]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

The capital of The Moon is **Lunaris Prime**.

Lunaris Prime is a sprawling, domed metropolis located in the Sea of Tranquility, chosen for its historical significance as the site of humanity's first lunar landing. The city is a marvel of engineering, with its protective domes made of advanced transparent alloys that allow residents to gaze out at the stars and Earth while shielding them from the harsh lunar environment.

The city is divided into three main sectors:

1. **Tranquility Hub**: The political and cultural heart of Lunaris Prime, housing the Lunar Council, the governing body of The Moon. This sector also features the Unity Spire, a towering structure that serves as a symbol of cooperation between Earth nations and lunar settlers.

2. **Stellar District**: The economic and technological center, home to research labs, mining corporations, and spaceport facilities. This district is powered by a network of solar farms spread across the lunar surface, providing clean energy to th

## Image input

In [11]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [12]:
print(uploader.value)

({'name': 'image.png', 'type': 'image/png', 'size': 151244, 'content': <memory at 0x000001231B34A200>, 'last_modified': datetime.datetime(2025, 12, 19, 17, 32, 49, 5000, tzinfo=datetime.timezone.utc)},)


In [13]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [14]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

This is a Talent Scorecard dashboard from Providence, which provides an overview of workforce metrics for a selected organization. Here's a breakdown of the key elements:

### 1. **Filters**
   - The dashboard allows filtering by:
     - **Pillar Head**
     - **Business Unit**
     - **Pillar**
     - **Core Leader**
     - **Job Family**
     - **Sub-Job Family**
     - **Date** (set to 12/19/2025 in this example)

### 2. **Key Metrics**
   - **Overall Headcount**: The total number of employees is 13.
   - **Direct Reports**: The number of employees directly reporting to a leader is also 13.
   - **Span of Control**: The average number of direct reports per leader is 13.00.

### 3. **Tenure by Grade**
   - This bar chart shows the tenure (experience in months) of employees by grade:
     - **1P**: 1 employee with tenure above 24 months.
     - **2P**: 1 employee with tenure between 7-12 months, and 4 employees with tenure above 24 months.
     - **3P**: 4 employees with tenure above 

## Audio input

In [15]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 50/50 [00:05<00:00,  9.83it/s]

Done.


In [17]:
agent = create_agent(
    model=get_azure_openai_model(),
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

BadRequestError: Error code: 400 - {'error': {'message': "Invalid 'messages[0]'. Content blocks are expected to be either text or image_url type.", 'type': 'invalid_request_error', 'param': 'messages[0]', 'code': 'invalid_value'}}